# 🎓 수학교육평가론 — 협력학습 MVP (v3)

사용자 1명 + AI 학생 2명의 협력학습 세션. 학습자모델·교수자모델을 구현하고 Gradio로 시각화.

```
User ──▶ 학습자 분석 LLM ──▶ Learner Model ──▶ Tutor Model ──▶ AI 학생 발화 LLM
                                                                         │
                                                              2 AI 학생 (민준·서연)
```

모든 로직은 **`tobagi_v1_0/lib/` 패키지**에 모듈화되어 있고, 이 노트북은 경로 설정 + 부트스트랩 + UI 기동만 담당합니다.

## 1️⃣ 의존성 설치 + GitHub 저장소 클론

- 최초 1회: `git clone`
- 재실행 시: 기존 `tobagi_v1_0/` 폴더가 있으면 `git pull`로 자동 갱신
- **한글 폰트(NanumGothic)는 `tobagi_v1_0/fonts/` 안에 동봉**되어 별도 설치 불필요

In [ ]:
# 셀 1: 라이브러리 + 코드 저장소
!pip install anthropic gradio -q

import os
REPO_URL = "https://github.com/hi-math/tobagi_v1.0.git"
if os.path.isdir("tobagi_v1_0"):
    !cd tobagi_v1_0 && git pull -q
    print("✅ tobagi_v1_0 저장소 업데이트 완료 (폰트 포함)")
else:
    !git clone -q $REPO_URL tobagi_v1_0
    print("✅ tobagi_v1_0 저장소 클론 완료 (NanumGothic 폰트 포함)")

## 2️⃣ 경로 · 모델 설정

**`BASE_PATH`** 와 **`MODEL`** 을 필요에 따라 수정하세요. 로컬 환경에서는 `BASE_PATH`를 리포지토리의 절대경로로 바꾸면 됩니다 (예: `/content/drive/MyDrive/team4`).

In [ ]:
# 셀 2: 경로 및 모델 설정
BASE_PATH = "tobagi_v1_0"                    # 클론된 리포지토리 루트 (상대/절대 모두 가능)
MODEL     = "claude-sonnet-4-20250514"       # Claude 모델 식별자

import sys, os
# lib 패키지를 import 가능하게: tobagi_v1_0 디렉토리를 sys.path에 추가
_abs = os.path.abspath(BASE_PATH)
if _abs not in sys.path:
    sys.path.insert(0, _abs)

print(f"✅ BASE_PATH={BASE_PATH!r}  (abs: {_abs})")
print(f"✅ MODEL={MODEL!r}")

## 3️⃣ 부트스트랩 — 설정 로드 + 학습자모델 초기화 + Claude API 준비

`bootstrap()` 한 번으로 `CONFIG`, `PROMPTS`, `learner_models`, `api` 네 가지를 dict로 받습니다. Gradio/CLI 런너에 `**ctx`로 그대로 주입할 수 있습니다.

In [ ]:
# 셀 3: 부트스트랩
from google.colab import userdata
from lib import bootstrap

ctx = bootstrap(
    base_path=BASE_PATH,
    api_key=userdata.get("CLAUDE_API_KEY"),
    model=MODEL,
    setup_fonts=True,        # NanumGothic 폰트 matplotlib 등록
)

# 편의용: 주요 객체를 노트북 전역에 펼쳐두기
CONFIG         = ctx["config"]
PROMPTS        = ctx["prompts"]
learner_models = ctx["learner_models"]
api            = ctx["api"]

print(f"✅ Task: {CONFIG['tasks']['task_title']}")
print(f"✅ Stages: {len(CONFIG['tasks']['stages'])}")
print(f"✅ AI 학생: {[p['name'] for p in CONFIG['personas']['ai_students'].values()]}")
print(f"✅ 학습자 모델 영역: {list(CONFIG['learner_model_schema']['models'].keys())}")
print(f"✅ 프롬프트: {list(PROMPTS.keys())}")

## 4️⃣ Gradio 대화형 UI 실행

- **왼쪽**: 채팅(나 ↔ 민준·서연) + `▶ 다음 Stage` 버튼
- **오른쪽 탭**: 🕸️ 레이더 · 📈 변화 추이 · 🧠 학습자모델 · 🧭 교수자 Decision
- Colab에서는 `share=True`로 공용 링크(~72시간) 생성

In [ ]:
# 셀 4: Gradio UI 기동
from lib import launch_ui

launch_ui(**ctx, share=True)

---

## 🗂️ (선택) 레거시 CLI 런너

Gradio UI가 정상 동작하면 실행할 필요 없음. 디버깅이나 오프라인 점검용.

명령어: `/radar` `/history` `/model` `/next` `/decision` `/quit`

In [ ]:
# 셀 5 (선택): CLI 세션
from lib import run_session

# run_session(**ctx)   # 주석 해제 후 실행